# Reddit eWOM — r/gaming Hype Scraper

For each unique game in `../games_master_player_counts.csv`, search r/gaming top posts (all time) where the game's name appears in the title. Keeping only posts created between **2021-01-01 and 2025-12-31**. 

Results:
- `games_reddit.csv` — one row per game (engagement, valence, n_posts)
- `reddit_posts.csv` — one row per matching post (title, score, upvote_ratio, num_comments, created_utc, permalink)

Using Reddit's public JSON endpoints, with a 1.1s delay between requests to stay under 60/min.

## 1. Setup

In [ ]:
import re
import time
import requests
import pandas as pd
from pathlib import Path

USER_AGENT = "SoCoFinalProject/0.1 by /u/CalendarExotic9775"
GAMES_CSV = Path("../games_with_aliases.csv")
OUTPUT_CSV = Path("games_reddit.csv")
POSTS_CSV = Path("reddit_posts.csv")
SEARCH_URL = "https://www.reddit.com/r/gaming/search.json"

REQUEST_DELAY = 1.1
TOP_N_FOR_VALENCE = 50
MAX_GAMES = 1923      

DATE_MIN_UTC = pd.Timestamp("2021-01-01").timestamp()
DATE_MAX_UTC = pd.Timestamp("2026-01-01").timestamp()

ALIASES = {}

## 2. Load Games + Aliases



In [ ]:
games_list = pd.read_csv(GAMES_CSV)
games_list = games_list.sort_values("peak_players", ascending=False).reset_index(drop=True)

ALIASES = {}
for _, row in games_list.iterrows():
    raw = row.get("aliases")
    if isinstance(raw, str) and raw.strip():
        ALIASES[row["game_name"]] = [a.strip() for a in raw.split("|") if a.strip()]

print(f"Unique games loaded:        {len(games_list):,}")
print(f"Games with ≥1 alias:        {sum(1 for v in ALIASES.values() if v):,}")
print(f"Total alias entries:        {sum(len(v) for v in ALIASES.values()):,}")
print(f"Will process top {MAX_GAMES} by peak_players")

games_list = games_list.head(MAX_GAMES)
games_list[["app_id", "game_name", "peak_players", "num_aliases", "aliases"]].head(10)

Unique games loaded:        1,923
Games with ≥1 alias:        150
Total alias entries:        201
Will process top 1923 by peak_players


,app_id,game_name,peak_players,num_aliases,aliases
0,751780,Forager,999,0,NaN
1,8500,EVE Online,999,0,NaN
2,628070,Romance of the Three Kingdoms XI with Power Up...,999,0,NaN
3,368500,Assassin's Creed Syndicate,999,0,NaN
4,366220,Wurm Unlimited,999,0,NaN
5,500,Left 4 Dead,999,0,NaN
6,1672740,崩壊3rd,999,0,NaN
7,1118200,People Playground,999,0,NaN
8,838350,太吾绘卷 The Scroll Of Taiwu,999,0,NaN
9,1313860,EA SPORTS™ FIFA 21,999,1,FIFA 21


## 3. Reddit Search Helper

Hits r/gaming's search endpoint with the game name as an exact-phrase query, sorted by top of all time. Returns the raw post records from the JSON response.


In [3]:
def _strip_symbols_and_quotes(s: str) -> str:
    s = re.sub(r"[™®©℠]", "", s)
    s = s.replace("‘", "'").replace("’", "'")
    s = s.replace("“", '"').replace("”", '"')
    return s

def normalize_name(s: str) -> str:
    """Normalize a game name (used for search query and regex pattern). Strips parentheticals like '(2007)'."""
    if not s:
        return ""
    s = _strip_symbols_and_quotes(s)
    s = re.sub(r"\s*\([^)]*\)", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_title(s: str) -> str:
    """Normalize a Reddit post title. Keeps parentheticals because the game name may be inside them."""
    if not s:
        return ""
    s = _strip_symbols_and_quotes(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def search_reddit(name: str, limit: int = 100) -> list[dict]:
    params = {
        "q": f'"{normalize_name(name)}"',
        "restrict_sr": "1",
        "sort": "top",
        "t": "all",
        "limit": str(limit),
    }
    headers = {"User-Agent": USER_AGENT}

    for attempt in range(3):
        r = requests.get(SEARCH_URL, params=params, headers=headers, timeout=20)
        if r.status_code == 200:
            data = r.json()
            return [child["data"] for child in data.get("data", {}).get("children", [])]
        if r.status_code == 429:
            time.sleep(5 * (attempt + 1))
            continue
        r.raise_for_status()
    return []

## 4. Title Matching

A post counts as a match only if the game's name appears as a whole word in the title. 

In [ ]:
def title_matches(title: str, name: str) -> bool:
    pattern = re.escape(normalize_name(name))
    return re.search(pattern, normalize_title(title), flags=re.IGNORECASE) is not None

## 5. Per-game Metrics

For one game: build a list of search terms (the dataset name + any aliases from the `ALIASES` dict), hit Reddit's search for each, deduplicate posts by id, keep only those whose title matches one of the variants (word-boundary, normalized), and only those created within the 2021–2025 window. Then compute engagement (sum of upvotes) and valence (mean upvote_ratio of the top 50 by upvotes).

Normalization strips `™ ® © ℠`, unifies curly quotes, drops parentheticals like `(2007)`, and collapses whitespace, so e.g. `Age of Empires III (2007)` will match a Reddit title like `Just played Age of Empires III last night`.

Also returns the raw matched posts so the main loop can save per-post records (title, score, upvote ratio, comment count, post date, permalink) to a separate CSV.

In [5]:
def in_window(post: dict) -> bool:
    ts = post.get("created_utc")
    return ts is not None and DATE_MIN_UTC <= ts < DATE_MAX_UTC

def compute_metrics(name: str) -> dict:
    search_terms = [name] + ALIASES.get(name, [])

    seen_ids = set()
    all_posts = []
    for j, term in enumerate(search_terms):
        if j > 0:
            time.sleep(REQUEST_DELAY)
        for p in search_reddit(term):
            pid = p.get("id")
            if pid and pid not in seen_ids:
                seen_ids.add(pid)
                all_posts.append(p)

    matched = [
        p for p in all_posts
        if any(title_matches(p.get("title", ""), t) for t in search_terms)
        and in_window(p)
    ]

    if not matched:
        return {"engagement": 0, "valence": float("nan"), "n_posts": 0, "posts": []}

    engagement = sum(p.get("score", 0) for p in matched)
    top = sorted(matched, key=lambda p: p.get("score", 0), reverse=True)[:TOP_N_FOR_VALENCE]
    ratios = [p.get("upvote_ratio", 0.0) for p in top if p.get("upvote_ratio") is not None]
    valence = sum(ratios) / len(ratios) if ratios else float("nan")

    return {
        "engagement": engagement,
        "valence": valence,
        "n_posts": len(matched),
        "posts": matched,
    }

## 6. Testrun



In [6]:
for name in games_list["game_name"].head(20):
    m = compute_metrics(name)
    print(f"{name:<40}  engagement={m['engagement']:>7}  valence={m['valence']:.3f}  n={m['n_posts']}")
    time.sleep(REQUEST_DELAY)

Forager                                   engagement=      3  valence=0.545  n=2
EVE Online                                engagement=  18130  valence=0.915  n=4
Romance of the Three Kingdoms XI with Power Up Kit  engagement=      0  valence=nan  n=0
Assassin's Creed Syndicate                engagement=   1537  valence=0.810  n=5
Wurm Unlimited                            engagement=      0  valence=nan  n=0
Left 4 Dead                               engagement= 162238  valence=0.905  n=13
崩壊3rd                                     engagement=      0  valence=nan  n=0
People Playground                         engagement=     50  valence=0.686  n=9
太吾绘卷 The Scroll Of Taiwu                  engagement=      0  valence=nan  n=0
EA SPORTS™ FIFA 21                        engagement=    508  valence=0.599  n=22
The Elder Scrolls IV: Oblivion Game of the Year Edition (2009)  engagement= 581311  valence=0.917  n=54
Plants vs. Zombies: Battle for Neighborville™  engagement=      7  valence=0.597  

## 7. Full Run

Loop over all selected games. Sleeps `REQUEST_DELAY` seconds between requests.

Results:
- `games_2023_reddit.csv` — one row per game with `engagement`, `valence`, `n_posts`.
- `reddit_posts.csv` — one row per matching post with `app_id`, `game_name`, `title`, `score`, `upvote_ratio`, `num_comments`, `created_utc`, `permalink`.



In [7]:
def safe_read_csv(path):
    """Read CSV, return empty DataFrame if file is missing or empty."""
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

done = safe_read_csv(OUTPUT_CSV)
if not done.empty:
    done_names = set(done["game_name"])
    results = done.to_dict(orient="records")
    print(f"Resuming — {len(done)} games already done.")
else:
    done_names = set()
    results = []

posts_existing = safe_read_csv(POSTS_CSV)
posts_records = posts_existing.to_dict(orient="records") if not posts_existing.empty else []
if posts_records:
    print(f"Loaded {len(posts_records)} existing post records.")

for i, row in games_list.iterrows():
    name = row["game_name"]
    if name in done_names:
        continue

    try:
        m = compute_metrics(name)
    except Exception as e:
        print(f"  [{i}] {name}: error {e}")
        m = {"engagement": None, "valence": None, "n_posts": None, "posts": []}

    for p in m.pop("posts", []):
        posts_records.append({
            "app_id": row.get("app_id"),
            "game_name": name,
            "title": p.get("title"),
            "score": p.get("score"),
            "upvote_ratio": p.get("upvote_ratio"),
            "num_comments": p.get("num_comments"),
            "created_utc": pd.to_datetime(p.get("created_utc"), unit="s", errors="coerce"),
            "permalink": "https://www.reddit.com" + (p.get("permalink") or ""),
        })

    record = row.to_dict()
    record.update(m)
    results.append(record)

    if (i + 1) % 10 == 0:
        print(f"  [{i+1:>3}/{len(games_list)}] {name[:30]:<30} engagement={m['engagement']}  n={m['n_posts']}")

    if (i + 1) % 25 == 0:
        pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
        pd.DataFrame(posts_records).to_csv(POSTS_CSV, index=False)

    time.sleep(REQUEST_DELAY)

pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
pd.DataFrame(posts_records).to_csv(POSTS_CSV, index=False)
print(f"\nDone. Saved {len(results)} games to {OUTPUT_CSV.name} and {len(posts_records)} posts to {POSTS_CSV.name}.")

  [ 10/1923] EA SPORTS™ FIFA 21             engagement=510  n=22
  [ 20/1923] MapleStory                     engagement=86  n=6
  [ 30/1923] Cube Racer                     engagement=0  n=0
  [ 40/1923] Verdun                         engagement=125  n=5
  [ 50/1923] A Plague Tale: Innocence       engagement=403  n=4
  [ 60/1923] DRAGON BALL FighterZ           engagement=3268  n=7
  [ 70/1923] Subsistence                    engagement=290  n=5
  [ 80/1923] Ravenfield                     engagement=6  n=3
  [ 90/1923] Pavlov                         engagement=1492  n=9
  [100/1923] Dishonored                     engagement=53233  n=8
  [110/1923] Secret Neighbor                engagement=0  n=0
  [120/1923] Warhammer 40,000: Space Marine engagement=0  n=0
  [130/1923] Stronghold Crusader HD         engagement=0  n=0
  [140/1923] Batman™: Arkham Origins        engagement=5750  n=4
  [150/1923] Cyber Manhunt                  engagement=0  n=0
  [160/1923] Hydroneer                      eng

## 8. Results

In [8]:
results_df = pd.read_csv(OUTPUT_CSV)

print(f"=== Games-level ({OUTPUT_CSV.name}) ===")
print(f"Total: {len(results_df)} games")
print(f"With at least one matching post: {(results_df['n_posts'] > 0).sum()}")
print(f"Mean engagement: {results_df['engagement'].mean():.1f}")
print(f"Mean valence:    {results_df['valence'].mean():.3f}")
display(results_df[["app_id", "game_name", "peak_players", "engagement", "valence", "n_posts"]].head(10))

if POSTS_CSV.exists():
    posts_df = pd.read_csv(POSTS_CSV, parse_dates=["created_utc"])
    print(f"\n=== Posts-level ({POSTS_CSV.name}) ===")
    print(f"Total posts: {len(posts_df)}")
    print(f"Date range:  {posts_df['created_utc'].min()}  →  {posts_df['created_utc'].max()}")
    print(f"Mean comments per post: {posts_df['num_comments'].mean():.1f}")
    display(posts_df[["game_name", "title", "score", "num_comments", "upvote_ratio", "created_utc"]].head(10))

=== Games-level (games_reddit.csv) ===
Total: 1923 games
With at least one matching post: 785
Mean engagement: 9877.3
Mean valence:    0.746


,app_id,game_name,peak_players,engagement,valence,n_posts
0,751780,Forager,999,2.0,0.530000,2.0
1,8500,EVE Online,999,18131.0,0.915000,4.0
2,628070,Romance of the Three Kingdoms XI with Power Up...,999,0.0,NaN,0.0
3,368500,Assassin's Creed Syndicate,999,1533.0,0.806000,5.0
4,366220,Wurm Unlimited,999,0.0,NaN,0.0
5,500,Left 4 Dead,999,162241.0,0.905385,13.0
6,1672740,崩壊3rd,999,0.0,NaN,0.0
7,1118200,People Playground,999,51.0,0.695556,9.0
8,838350,太吾绘卷 The Scroll Of Taiwu,999,0.0,NaN,0.0
9,1313860,EA SPORTS™ FIFA 21,999,510.0,0.597273,22.0



=== Posts-level (reddit_posts.csv) ===
Total posts: 6502
Date range:  2021-01-01 15:30:20  →  2025-12-31 19:52:58
Mean comments per post: 156.3


,game_name,title,score,num_comments,upvote_ratio,created_utc
0,Forager,Looking for a game like forager,2,2,0.56,2024-09-13 02:51:04
1,Forager,Forager Co-op,0,9,0.50,2022-09-24 14:22:57
2,EVE Online,The most expensive gameing war was unsurprisin...,13285,740,0.95,2022-05-16 23:37:40
3,EVE Online,One of the servers that powered EVE Online bac...,4378,150,0.98,2023-11-09 14:52:30
4,EVE Online,EVE Online | Down The Rabbit Hole | A 6 hour d...,330,57,0.87,2023-11-01 19:10:58
5,EVE Online,Latest Alliance Theft in EVE Online: Multiple ...,138,65,0.86,2023-11-21 22:21:54
6,Assassin's Creed Syndicate,Assassin’s Creed Syndicate 60 FPS/4k Console P...,928,129,0.94,2024-11-18 18:57:11
7,Assassin's Creed Syndicate,Assassin's Creed Syndicate is definetly one of...,511,252,0.73,2023-07-02 07:18:09
8,Assassin's Creed Syndicate,Started playing Assassin's Creed Syndicate and...,66,47,0.81,2022-06-03 15:02:07
9,Assassin's Creed Syndicate,i was playing assassin's Creed syndicate earli...,18,2,0.89,2022-10-04 14:52:51
